In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torchvision import datasets, transforms
from torchvision.models.resnet import ResNet, BasicBlock
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


# Group 1

In [4]:
transform = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914,0.4822,0.4465],
                         std=[0.247,0.243,0.261])
])

# Load CIFAR-10 dataset
train_dataset_full = datasets.CIFAR10(root='./cifar-10-batches-py', train=True,
                                      download=True, transform=transform)
test_dataset_full = datasets.CIFAR10(root='./cifar-10-batches-py', train=False,
                                     download=True, transform=transform)


Files already downloaded and verified
Files already downloaded and verified


In [5]:
cat_class = 3
dog_class = 5

targets_array = np.array(train_dataset_full.targets)
indices = np.where((targets_array == cat_class) | (targets_array == dog_class))[0]

filtered_images = train_dataset_full.data[indices]
filtered_labels = np.array([1 if lbl == cat_class else 0 for lbl in targets_array[indices]])

In [6]:
class CatDogDataset(Dataset):
    def __init__(self, images, labels):
        self.images = images
        self.targets = labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.targets[idx]

In [7]:
train_dataset = CatDogDataset(filtered_images, filtered_labels)

In [8]:
targets_array_test = np.array(test_dataset_full.targets)
indices_test = np.where((targets_array_test == cat_class) | (targets_array_test == dog_class))[0]

filtered_images_test = test_dataset_full.data[indices_test]
filtered_labels_test = np.array([1 if lbl == cat_class else 0 for lbl in targets_array_test[indices_test]])

test_dataset = CatDogDataset(filtered_images_test, filtered_labels_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("train - test dataset ready.")

train - test dataset ready.


# Group 2

In [10]:
# --- Create the Test Set and Loader ---
print("Creating the test set...")

# Get the raw test data (this variable was loaded by Group 1)
test_targets_np = np.array(test_data.targets)

# Find the indices for cats (3) and dogs (5)
test_cat_dog_indices = np.where((test_targets_np == 3) | (test_targets_np == 5))[0]

# Filter the data and labels
test_filtered_images = test_data.data[test_cat_dog_indices]
test_filtered_labels_raw = test_targets_np[test_cat_dog_indices]

# Remap labels: cat (3) -> 1, dog (5) -> 0
test_filtered_labels = np.where(test_filtered_labels_raw == 3, 1, 0)

# Create the test dataset instance
cat_dog_test = CatDogDataset(test_filtered_images, test_filtered_labels, transform=transform)

print(f"Test set created with {len(cat_dog_test)} images.")

# Create the Test Loader
BATCH_SIZE = 64 # Using the same batch size as the train_loader
test_loader = DataLoader(cat_dog_test, batch_size=BATCH_SIZE, shuffle=False)

print(f"Test loader created with batch size {BATCH_SIZE}.")
print("All data is now loaded. Ready to build the model.")

Creating the test set...


NameError: name 'test_data' is not defined

In [ ]:
# --- Load, Modify, and Freeze ResNet18 ---

print("\nLoading pretrained ResNet18 model...")
# We use the recommended 'DEFAULT' weights (ImageNetV1)
model = resnet18(weights=ResNet18_Weights.DEFAULT)


# --- Modify the final FC layer for binary classification ---
# Get the number of input features for the original 'fc' layer
num_ftrs = model.fc.in_features
# Replace the final layer with a new one that has only 1 output neuron
model.fc = nn.Linear(num_ftrs, 1)
print(f"ResNet18 final 'fc' layer modified for 1 output.")


# --- Freeze all layers EXCEPT layer3, layer4, and fc ---
# First, freeze all parameters in the model
for param in model.parameters():
    param.requires_grad = False

# Now, unfreeze *only* the layers you need to fine-tune
print("Unfreezing layers for fine-tuning: 'layer3', 'layer4', and 'fc'")
for param in model.layer3.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True
    

# --- Verification Step ---
print("\n--- Trainable Parameters Verification ---")
total_params = 0
trainable_params = 0

for name, param in model.named_parameters():
    total_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()
        print(f"  [TRAINABLE] {name}")

print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

print("\nModel setup is complete.")
print("The real data is loaded and the model is ready for fine-tuning.")